# Open-source cardiac MR Fingerprinting - In-vivo application
This notebook shows how to do image reconstruction and parameter estimation for the travelling volunteer study.

## Overview
Cardiac MR Fingerprinting (cMRF) data was acquired in three volunteers who were scanned at two different scanners. In this notebook we will reconstruct the images and estimate the $T_1$ and $T_2$ maps. In addition to the cMRF data, also Cartesian and golden radial cine data was acquired. We will also reconstruct that to verify the anatomical features seen in the quantitative maps.

This notebook utilises MRpro for reconstruction and parameter estimation: https://github.com/PTB-MR/mrpro and it is designed to be run via Google colab: 

<a target="_blank" href="https://colab.research.google.com/github/PTB-MR/cMRF/blob/main/open_source_cmrf_in_vivo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Outline

1) Installation of MRpro

2) Define image reconstruction and parameter estimation methods for cMRF and cine sequences

3) Select the volunteer and scanner

4) Download the data from zenodo

5) Reconstruct images and calculate $T_1$ and $T_2$ maps

6) Visualise results

### 1) Installation of MRpro
Install MRpro (currently a working github-branch but will be changed to pypi package as soon as possible) and import

In [ ]:
!pip install git+https://github.com/PTB-MR/mrpro.git@main#egg=mrpro[notebooks]

In [ ]:
# Imports
import numpy as np
import torch
import matplotlib.pyplot as plt
import tempfile
import os
import glob
from cmap import Colormap

import zenodo_get
from matplotlib.colors import ListedColormap
from pathlib import Path
from mrpro.data import KData, IData, CsmData, SpatialDimension 
from mrpro.algorithms.reconstruction import DirectReconstruction, IterativeSENSEReconstruction
from mrpro.operators.models import CardiacFingerprinting
from mrpro.data.traj_calculators import KTrajectoryCartesian, KTrajectoryIsmrmrd
from mrpro.operators import AveragingOp, DictionaryMatchOp
    

### 2) Define image reconstruction and parameter estimation methods for cMRF and cine sequences

#### Cartesian cine reconstruction
This data was acquired using a random Cartesian undersampling scheme, where the undersampling pattern varies between 
the different cardiac phases. The acquisition is prospectively triggered and data for 30 cardiac phases is acquired. 
The k-space center is fully acquired. 

Image reconstruction is carried out by:
- Reconstruct coil-resolved cine data using simply FFT
- Averaging the images along the cardiac phase dimension
- Estimating the coil sensitivity maps (csm) from the averaged images
- Reconstruct the final cine data using the csm and an iterative SENSE approach

In [ ]:
def cart_cine_reconstruction(kdata_fname):
    """Reconstruct a cartesian cine."""
    kdata = KData.from_file(kdata_fname, KTrajectoryCartesian())
    kdata.header.recon_matrix = SpatialDimension(z=1, y=160, x=160)
    idat = DirectReconstruction(kdata, csm=None)(kdata)
    idat_phase_average = IData.from_tensor_and_kheader(data=idat.data.mean(dim=0)[None,...], header=kdata.header[0,...])
    csm = CsmData.from_idata_inati(idat_phase_average, downsampled_size=64)
    
    reco = IterativeSENSEReconstruction(kdata, csm=csm)
    idat_cart_cine = reco(kdata).data.abs().squeeze().numpy()
    idat_cart_cine = idat_cart_cine/idat_cart_cine.max()
    return idat_cart_cine
          

#### Radial cine reconstruction
This data was acquired using a Golden radial sampling scheme. The acquisition is prospectively triggered and data is 
acquired continuously in several cardiac cycles. The trajectory is part of the ISMRMRD file. 

Image reconstruction is carried out by:
- Reconstruct an average image using all the acquired data
- Estimating the coil sensitivity maps (csm) from the average image
- Split the data into 30 cardiac cycles
- Reconstruct the final cine data using the csm and an iterative SENSE approach

In [ ]:
def radial_cine_reconstruction(kdata_fname):
    """Reconstruct a radial cine."""
    kdata = KData.from_file(kdata_fname, KTrajectoryIsmrmrd())
    kdata.header.recon_matrix = SpatialDimension(z=1, y=160, x=160)
    csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
    
    # Data is acquired continously. Here we are resorting the data and combining radial lines obtained in the same 
    # cardiac phase but different cardiac cycle. 
    n_phases = 30
    n_lines_per_phase = 7
    n_cycles = 16
    n_lines_per_cycle = kdata.shape[-2]//n_cycles
    
    idx_first_phase = torch.cat([torch.arange(n_lines_per_cycle*cycle_idx, n_lines_per_phase+n_lines_per_cycle*cycle_idx) for cycle_idx in range(n_cycles)])
    split_indices = torch.stack([idx_first_phase + n_lines_per_phase*phase for phase in range(n_phases)])
    kdata_split = kdata[..., split_indices, :]
    
    reco = IterativeSENSEReconstruction(kdata_split, csm=csm)
    idat_rad_cine = reco(kdata_split).data.abs().squeeze().numpy()
    idat_rad_cine = idat_rad_cine/idat_rad_cine.max()
    return idat_rad_cine

#### T1 and T2 mapping
This data was acquired using the open-source cardiac MRF approach. For more information on the reconstruction please 
have a look at the notebook "open_source_cmrf_scanner_comparison.ipynb", which is part of this repo and the exmaple notebook
"" in MRpro. 
 
T1 and T2 mapping is carried out by:
- Reconstruct an average image using all the acquired data
- Estimate the coil sensitivity maps (csm) from the average image
- Split the data into different dynamics
- Reconstruct the dynamic images using the csm and an iterative SENSE approach
- Estimate fingerprints for a range of expected T1 and T2 values using an extended phase graph model
- Calculate T1 and T2 maps by comparing the reconstructed dynamic images to the dictionary on a voxel-by-voxel basis


In [ ]:
def t1_t2_cmrf_mapping(kdata_fname):
    """Reconstruct T1 and T2 maps."""
    kdata = KData.from_file(kdata_fname, KTrajectoryIsmrmrd())
    kdata.header.recon_matrix = SpatialDimension(z=1, y=180, x=180)
    csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
    
    n_acq_per_image = 20
    n_overlap = 10
    n_acq_per_block = 47
    n_blocks = 15

    idx_in_block = torch.arange(n_acq_per_block).unfold(0, n_acq_per_image, n_acq_per_image - n_overlap)
    split_indices = (n_acq_per_block * torch.arange(n_blocks)[:, None, None] + idx_in_block).flatten(end_dim=1)
    kdata_split = kdata[..., split_indices, :]
    
    reco = IterativeSENSEReconstruction(kdata_split, csm=csm, n_iterations=10)
    idat = reco(kdata_split)
    
    model = AveragingOp(dim=0, idx=split_indices) @ CardiacFingerprinting(
        kdata.header.acq_info.acquisition_time_stamp.squeeze(),
        echo_time=kdata.header.te[0],
        repetition_time=kdata.header.tr[0],
        t2_prep_echo_times=(0.03, 0.05, 0.08),
    )
    dictionary = DictionaryMatchOp(model, index_of_scaling_parameter=0)
    t1_keys = torch.arange(0.05, 2, 0.05)[:, None]
    t2_keys = torch.arange(0.01, 0.15, 0.01)[None, :]
    m0_keys = torch.tensor(1.0)
    dictionary.append(m0_keys, t1_keys, t2_keys)
    m0_match, t1_match, t2_match = dictionary(idat.data[:, 0, 0, 0])
    return m0_match, t1_match, t2_match
    

### 3) Select the volunteer and scanner

In this study we obtained data from three volunteers (vol1, vol2, vol3) on two scanners (scanner1, scanner3). We kept 
the naming convention here as it is in the paper. For one scan only the cMRF data had been acquired. 
Here you can set the volunteer index and the scanner index:

In [ ]:
volunteer_index = 2 # should be 1,2 or 3
scanner_index = 1 # should be 1 or 3

### 4) Download the data from zenodo

In [ ]:
assert volunteer_index > 0 and volunteer_index < 4, 'Volunteer index should be 1,2 o 3'
assert scanner_index == 1 or scanner_index == 3, 'Scanner index should be 1 or 3'


tmp = tempfile.TemporaryDirectory()  # RAII, automatically cleaned up
data_folder = Path(tmp.name)
zenodo_get.download(record='15831511', retry_attempts=5, output_dir=data_folder, file_glob=(f'*scanner{scanner_index}_vol{volunteer_index}*',))


### 5) Reconstruct images and calculate $T_1$ and $T_2$ maps

In [ ]:


cart_cine = glob.glob(os.path.join(data_folder, f'scanner{scanner_index}_vol{volunteer_index}_cartesian_cine.mrd'))
if len(cart_cine) > 0:
    idat_cart_cine = cart_cine_reconstruction(cart_cine[0])


rad_cine = glob.glob(os.path.join(data_folder, f'scanner{scanner_index}_vol{volunteer_index}_radial_cine.mrd'))
if len(rad_cine) > 0:
    idat_rad_cine = radial_cine_reconstruction(rad_cine[0])
    

cmrf = glob.glob(os.path.join(data_folder, f'scanner{scanner_index}_vol{volunteer_index}_cmrf.mrd'))
if len(cmrf) > 0:
    m0_match, t1_match, t2_match = t1_t2_cmrf_mapping(cmrf[0])
    
if (scanner_index == 3 and volunteer_index != 1) or volunteer_index == 3:
    fliplr = True
else:
    fliplr = False
if scanner_index == 3 and volunteer_index == 3:
    rot90_value = -2
else:
    rot90_value = -1
    
def orientate_image(idat, rot90_value, fliplr):
    idat = np.fliplr(idat) if fliplr else idat
    return(np.rot90(idat, rot90_value))

### 6) Visualise and evaluate results

Now we visualise and compare all the results.

In [ ]:
fig, ax = plt.subplots(2,3, squeeze=False, figsize=(12,8))
[cax.set_xticks([]) for cax in ax.flatten()]
[cax.set_yticks([]) for cax in ax.flatten()]
if len(cart_cine) > 0:
    ax[0,0].imshow(orientate_image(idat_cart_cine[0,...], rot90_value, fliplr), vmax=0.8, cmap='gray')
    ax[0,0].set_title('Cartesian cine (diastole)')
    ax[1,0].imshow(orientate_image(idat_cart_cine[10,...], rot90_value, fliplr), vmax=0.8, cmap='gray')
    ax[1,0].set_title('Cartesian cine (systole)')

if len(rad_cine) > 0:
    ax[0,1].imshow(orientate_image(idat_rad_cine[0,...], rot90_value, fliplr), vmax=0.6, cmap='gray')
    ax[0,1].set_title('Radial cine (diastole)')
    ax[1,1].imshow(orientate_image(idat_rad_cine[10,...], rot90_value, fliplr), vmax=0.6, cmap='gray')
    ax[1,1].set_title('Radial cine (systole)')

if len(cmrf) > 0:
    ax[0,2].imshow(orientate_image(t1_match, rot90_value, fliplr), vmin=0.2, vmax=2, cmap=Colormap('lipari').to_mpl())
    ax[0,2].set_title('T1 map')
    ax[1,2].imshow(orientate_image(t2_match, rot90_value, fliplr), vmin=0., vmax=0.15, cmap=Colormap('navia').to_mpl())
    ax[1,2].set_title('T2 map')